In [ ]:
!pip install evaluate rouge-score nltk


In [ ]:
!pip install evaluate

In [ ]:
!pip install rouge_score

In [ ]:
!git clone https://github.com/salaniz/pycocoevalcap.git
!pip install pycocoevalcap

In [ ]:
!pip install bert-score

In [ ]:
import gc
import torch

# Clean Python memory
gc.collect()

# Clean CUDA memory if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("✔️ Cache cleared successfully!")

In [ ]:
from google.colab import drive
import sys

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/Models/ArQModel')

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torchvision.transforms as transforms
import torch.nn.functional as F
from tqdm import tqdm
import evaluate
import json
from nltk.translate.bleu_score import corpus_bleu

from the_datasets3 import *
from utils3 import *
from models3 import DecoderWithAttention, Attention


#Parameters

data_folder = "/content/drive/MyDrive/Models/ArQModel/FinalDataset"
data_name = 'coco_5_cap_per_img_5_min_word_freq'
checkpoint_file ='/content/drive/MyDrive/Models/ArQModel/Checkpoints_best/coco_5_cap_per_img_5_min_word_freq_BEST.pth.tar'
word_map_file = '/content/drive/MyDrive/Models/ArQModel/FinalDataset/WORDMAP_coco_5_cap_per_img_5_min_word_freq.json'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cudnn.benchmark = True


#Load Word Map
with open(word_map_file, 'r') as j:
    word_map = json.load(j)
rev_word_map = {v: k for k, v in word_map.items()}
vocab_size = len(word_map)


#Load Model (state_dict only)
torch.serialization.add_safe_globals([DecoderWithAttention, Attention])

checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=False)
print("Loaded checkpoint keys:", list(checkpoint.keys()))

decoder = DecoderWithAttention(
    attention_dim=1024,
    embed_dim=1024,
    decoder_dim=1024,
    vocab_size=len(word_map),
    dropout=0.5
).to(device)

decoder.load_state_dict(checkpoint["decoder_state_dict"])
decoder.eval()
print("Decoder weights loaded correctly and ready for evaluation.")


bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
rouge = evaluate.load("rouge")


def compute_metrics(references, hypotheses):
    results = {}
    results["Bleu_4"] = bleu.compute(predictions=hypotheses, references=[[r] for r in references])["bleu"]
    results["METEOR"] = meteor.compute(predictions=hypotheses, references=[[r] for r in references])["meteor"]
    results["ROUGE_L"] = rouge.compute(predictions=hypotheses, references=[[r] for r in references])["rougeL"]
    return results


#Evaluation Loop
def evaluate_model(beam_size=5):
    loader = torch.utils.data.DataLoader(
        CaptionDataset(data_folder, data_name, 'TEST'),
        batch_size=1, shuffle=True, num_workers=1, pin_memory=torch.cuda.is_available()
    )

    references, hypotheses = [], []
    img_idx_list = []  # جديييييييييييييد
    for i, (image_features, caps, caplens, allcaps, img_idx) in enumerate( # uuuuuuuuuuuuuuuu
            tqdm(loader, desc=f"EVALUATING AT BEAM SIZE {beam_size}")):

        img_idx_list.append(int(img_idx.item())) # جديييييييييييد


        k = beam_size
        image_features = image_features.to(device)
        image_features_mean = image_features.mean(1).expand(k, 2048)

        k_prev_words = torch.LongTensor([[word_map['<start>']]] * k).to(device)
        seqs = k_prev_words
        top_k_scores = torch.zeros(k, 1).to(device)
        complete_seqs, complete_seqs_scores = [], []
        step = 1

        h1, c1 = decoder.init_hidden_state(k)
        h2, c2 = decoder.init_hidden_state(k)

        while True:
            embeddings = decoder.embedding(k_prev_words).squeeze(1)
            h1, c1 = decoder.top_down_attention(
                torch.cat([h2, image_features_mean, embeddings], dim=1), (h1, c1)
            )
            attention_weighted_encoding = decoder.attention(image_features, h1)
            h2, c2 = decoder.language_model(
                torch.cat([attention_weighted_encoding, h1], dim=1), (h2, c2)
            )

            scores = decoder.fc(h2)
            scores = F.log_softmax(scores, dim=1)
            scores = top_k_scores.expand_as(scores) + scores

            if step == 1:
                top_k_scores, top_k_words = scores[0].topk(k, 0, True, True)
            else:
                top_k_scores, top_k_words = scores.view(-1).topk(k, 0, True, True)

            vocab_size_model = decoder.vocab_size
            prev_word_inds = top_k_words // vocab_size_model
            next_word_inds = top_k_words % vocab_size_model

            seqs = torch.cat([seqs[prev_word_inds], next_word_inds.unsqueeze(1)], dim=1)

            incomplete_inds = [
                ind for ind, next_word in enumerate(next_word_inds)
                if next_word != word_map['<end>']
            ]
            complete_inds = list(set(range(len(next_word_inds))) - set(incomplete_inds))

            if len(complete_inds) > 0:
                complete_seqs.extend(seqs[complete_inds].tolist())
                complete_seqs_scores.extend(top_k_scores[complete_inds])
            k -= len(complete_inds)

            if k == 0:
                break
            seqs = seqs[incomplete_inds]
            h1 = h1[prev_word_inds[incomplete_inds]]
            c1 = c1[prev_word_inds[incomplete_inds]]
            h2 = h2[prev_word_inds[incomplete_inds]]
            c2 = c2[prev_word_inds[incomplete_inds]]
            image_features_mean = image_features_mean[prev_word_inds[incomplete_inds]]
            top_k_scores = top_k_scores[incomplete_inds].unsqueeze(1)
            k_prev_words = next_word_inds[incomplete_inds].unsqueeze(1)

            if step > 50:
                break
            step += 1

        if len(complete_seqs_scores) == 0:
            complete_seqs = seqs.tolist()
            complete_seqs_scores = top_k_scores.squeeze(1).tolist()


        i_best = complete_seqs_scores.index(max(complete_seqs_scores))
        seq = complete_seqs[i_best]

        # Reference
        img_caps = allcaps[0].tolist()
        img_captions = list(map(lambda c: [
            rev_word_map[w] for w in c if w not in
            {word_map['<start>'], word_map['<end>'], word_map['<pad>']}
        ], img_caps))
        img_caps_texts = [' '.join(c) for c in img_captions]
        references.append(img_caps_texts)

        # Hypothesis
        hypothesis = [rev_word_map[w] for w in seq if w not in
                      {word_map['<start>'], word_map['<end>'], word_map['<pad>']}]
        hypotheses.append(' '.join(hypothesis))

    # Save all results
    # جدييييد
    results = [{
    "image_index": int(img_idx_list[i]),
    "reference": ref,
    "generated": hyp
    } for i, (ref, hyp) in enumerate(zip(references, hypotheses))]


    # قديييييييييييم
    #results = [{"reference": ref, "generated": hyp} for ref, hyp in zip(references, hypotheses)]

    output_path = "/content/drive/MyDrive/Models/ArQModel/eval_results.json"#remove _best
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {output_path}")

    # Compute metrics
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider

    gts = {i: refs for i, refs in enumerate(references)}
    res = {i: [hyp] for i, hyp in enumerate(hypotheses)}

    bleu_scorer = Bleu(4)
    bleu_score, _ = bleu_scorer.compute_score(gts, res)

    meteor_scorer = Meteor()
    meteor_score, _ = meteor_scorer.compute_score(gts, res)

    rouge_scorer = Rouge()
    rouge_score, _ = rouge_scorer.compute_score(gts, res)

    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(gts, res)

    print("\n Evaluation Metrics:")
    print(f"BLEU-1: {bleu_score[0]:.4f}")
    print(f"BLEU-2: {bleu_score[1]:.4f}")
    print(f"BLEU-3: {bleu_score[2]:.4f}")
    print(f"BLEU-4: {bleu_score[3]:.4f}")
    print(f"METEOR: {meteor_score:.4f}")
    print(f"ROUGE-L: {rouge_score:.4f}")
    print(f"CIDEr:  {cider_score:.4f}")

    return {
        "BLEU": bleu_score,
        "METEOR": meteor_score,
        "ROUGE_L": rouge_score,
        "CIDEr": cider_score
    }

#Run Evaluation
if __name__ == '__main__':
    metrics_dict = evaluate_model(beam_size=5)
    print(metrics_dict)


In [ ]:
# ===== Arabic QA Evaluation + Side-by-Side (with CIDEr + BERTScore) =====

import json, re, csv
from pathlib import Path
from statistics import mean
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# ---------- Optional libs ----------
HAVE_SACREBLEU = HAVE_BERTSCORE = True

try:
    from sacrebleu import corpus_bleu as sb_corpus_bleu
    from sacrebleu.metrics import CHRF
except Exception as e:
    print("[warn] sacrebleu not available ->", e)
    HAVE_SACREBLEU = False

try:
    from bert_score import score as bertscore_score
except Exception as e:
    print("[warn] bert-score not available ->", e)
    HAVE_BERTSCORE = False


# ---------- CIDEr ----------
def compute_cider_corpus(hyps, refs):
    try:
        from pycocoevalcap.cider.cider import Cider
        gts = {i: rs for i, rs in enumerate(refs)}
        res = {i: [h] for i, h in enumerate(hyps)}
        scorer = Cider()
        score, scores = scorer.compute_score(gts, res)
        return float(score), [float(s) for s in scores]
    except Exception as e:
        print("[warn] CIDEr not available ->", e)
        return None, []


# ---------- Paths ----------
baseline_path = Path("/content/drive/MyDrive/Models/ArQModel/eval_results.json")

out_report    = baseline_path.with_name("eval_full_metrics_ar.json")
out_side_json = baseline_path.with_name("eval_side_by_side.json")
out_side_csv  = baseline_path.with_name("eval_side_by_side.csv")


# ---------- Arabic normalization ----------
_diac = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
_ctrl = re.compile(r'[\u200e\u200f\u202a-\u202e]')

def norm(s):
    s = _ctrl.sub('', s or '')
    s = _diac.sub('', s)
    s = s.replace('ـ', '')
    s = re.sub('[إأآا]', 'ا', s)
    s = re.sub('ى', 'ي', s)
    return re.sub(r'\s+', ' ', s).strip()

def basic_tok(s):
    return re.findall(r'[\u0600-\u06FF]+|[A-Za-z]+|\d+', norm(s))

def ar_clitic_split(s):
    s = norm(s)
    s = re.sub(r'\b([وفبكل])([\u0621-\u064A])', r'\1 \2', s)
    s = re.sub(r'\bال([\u0621-\u064A])', r'ال \1', s)
    return s

def word_tokens_with_clitics(s):
    return ar_clitic_split(s).split()


# ---------- Extractors ----------
def first_non_empty_text(x):
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):
        for e in x:
            t = first_non_empty_text(e)
            if t:
                return t
    return ""

def many_texts_list(x):
    if isinstance(x, list):
        return [norm(t) for t in x if isinstance(t, str) and t.strip()]
    if isinstance(x, str):
        return [norm(x)]
    return []


# ---------- Load data ----------
data = json.loads(baseline_path.read_text(encoding="utf-8"))

cocoids, hyps_txt, refs_txt = [], [], []
kept = skipped = 0

for obj in data:
    cid = str(obj.get("image_index", "")).strip()
    hyp = first_non_empty_text(obj.get("generated"))
    refs = many_texts_list(obj.get("reference"))

    if not cid or not hyp or not refs:
        skipped += 1
        continue

    cocoids.append(cid)
    hyps_txt.append(norm(hyp))
    refs_txt.append([norm(r) for r in refs])
    kept += 1

print(f"Loaded: {len(data)} | Used: {kept} | Skipped: {skipped}")


# ---------- BLEU ----------
smooth = SmoothingFunction().method3
hyps_tok = [basic_tok(h) for h in hyps_txt]
refs_tok = [[basic_tok(r) for r in rs] for rs in refs_txt]

def bleu_n(n):
    weights = {
        1: (1,0,0,0),
        2: (0.5,0.5,0,0),
        3: (1/3,1/3,1/3,0),
        4: (0.25,0.25,0.25,0.25)
    }[n]
    return corpus_bleu(refs_tok, hyps_tok, weights=weights, smoothing_function=smooth)

bleu1, bleu2, bleu3, bleu4 = bleu_n(1), bleu_n(2), bleu_n(3), bleu_n(4)


# ---------- ROUGE-L ----------
def lcs_len(a, b):
    n, m = len(a), len(b)
    dp = [0]*(m+1)
    for i in range(1, n+1):
        prev = 0
        for j in range(1, m+1):
            tmp = dp[j]
            if a[i-1] == b[j-1]:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j-1])
            prev = tmp
    return dp[m]

def rougeL_char(h, r):
    L = lcs_len(h, r)
    return 0 if L == 0 else (2*L)/(len(h)+len(r))

rl_char = [max(rougeL_char(h, r) for r in rs) for h, rs in zip(hyps_txt, refs_txt)]
rougeL_char_mean = mean(rl_char)


# ---------- CIDEr ----------
cider_corpus, cider_per_item = compute_cider_corpus(hyps_txt, refs_txt)


# ---------- BERTScore (Recall + F1) ----------
bertscore_R = bertscore_F1 = None
if HAVE_BERTSCORE:
    best_refs = [max(rs, key=lambda r: rougeL_char(h, r)) for h, rs in zip(hyps_txt, refs_txt)]
    P, R, F1 = bertscore_score(
        hyps_txt, best_refs,
        lang="ar",
        model_type="xlm-roberta-large",
        rescale_with_baseline=True
    )
    bertscore_R  = float(R.mean())
    bertscore_F1 = float(F1.mean())


# ---------- Print ----------
print("\n=== Evaluation Metrics ===")
print(f"BLEU-1  : {bleu1:.4f}")
print(f"BLEU-2  : {bleu2:.4f}")
print(f"BLEU-3  : {bleu3:.4f}")
print(f"BLEU-4  : {bleu4:.4f}")
print(f"ROUGE-L : {rougeL_char_mean:.4f}")
print(f"CIDEr   : {cider_corpus:.4f}")
print(f"BERTScore Recall : {bertscore_R:.4f}")
print(f"BERTScore F1     : {bertscore_F1:.4f}")


# ---------- Save report ----------
report = {
    "counts": {"used": kept, "skipped": skipped},
    "metrics": {
        "BLEU_1": bleu1,
        "BLEU_2": bleu2,
        "BLEU_3": bleu3,
        "BLEU_4": bleu4,
        "ROUGE_L": rougeL_char_mean,
        "CIDEr": cider_corpus,
        "BERTScore_Recall": bertscore_R,
        "BERTScore_F1": bertscore_F1
    }
}

out_report.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSaved full report → {out_report}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, pickle

results_path = "/content/drive/MyDrive/Models/ArQModel2/eval_results.json"
pkl_path = "/content/drive/MyDrive/COCO/Features/all_ids.pkl"
output_path = "/content/drive/MyDrive/Models/ArQModel2/eval_results_with_ids.json"

if os.path.exists(results_path):
    print("File found:", results_path)

    # --- Load JSON results ---
    with open(results_path, 'r') as f:
        data = json.load(f)

    # --- Load mapping image_id -> index ---
    with open(pkl_path, "rb") as f:
        imageid_to_index = pickle.load(f)

    # --- Invert mapping to get index -> image_id ---
    index_to_imageid = {v: k for k, v in imageid_to_index.items()}

    # --- Replace image_index with image_id in all items ---
    for item in data:
        img_idx = item.get("image_index")
        img_id = index_to_imageid.get(img_idx, "UNKNOWN")
        item["image_id"] = img_id        # أضف/استبدل بالـ image_id
        item.pop("image_index", None)    # احذف الـ index القديم

    # --- Save new JSON file ---
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Updated JSON saved to: {output_path}")
    print(f"Total captions saved: {len(data)}")
    print("Showing first samples with image IDs:\n")

    # --- Print first 50 samples ---
    for i, item in enumerate(data[:4900]):
        print(f"Sample {i+1} | Image ID: {item['image_id']}")
        print("References:")
        for j, ref in enumerate(item['reference']):
            print(f"  Reference {j+1}: {ref}")

        print("Generated:")
        print(f"  {item['generated']}")
        print("-" * 40)

else:
    print("File not found")

In [ ]:
import os
import json
from IPython.display import Image, display

# ---- Paths ----
results_path = "/content/drive/MyDrive/Models/ArQModel2/eval_results_with_ids.json"

# مجلدات الصور
image_dirs = [
    "/content/drive/MyDrive/COCO/Images/val2014/",
    "/content/drive/MyDrive/COCO/Images/val2014 (1)/"
]

def find_image_path(image_id):
    """
    تبحث عن أي صورة ينتهي اسمها بـ image_id.jpg
    مع ضمان أن image_id = 6 digits (leading zeros)
    """
    # ---- إضافة الأصفار قبل البحث ----
    image_id = str(int(image_id)).zfill(6)

    for folder in image_dirs:
        for fname in os.listdir(folder):
            if fname.endswith(f"{image_id}.jpg"):
                return os.path.join(folder, fname)

    return None


# ---- Load JSON ----
with open(results_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded results:", len(data))
print("Displaying unique image results only...\n")

# ---- تتبع الصور المعروضة ----
shown_image_ids = set()

# ---- Display image + captions (once per image) ----
for item in data:
    img_id = item["image_id"]

    # إذا تكرر الإيميج → تخطي الصورة والكابشنات
    if img_id in shown_image_ids:
        continue

    shown_image_ids.add(img_id)

    img_path = find_image_path(img_id)

    print("=" * 70)
    print(f"Image ID: {img_id}")

    if img_path:
        display(Image(filename=img_path))
    else:
        print("IMAGE NOT FOUND:", img_id)

    print("\nGenerated:")
    print(" ", item["generated"])
    print("=" * 70)


In [ ]:
import os
import json
from IPython.display import Image, display

# ---- Paths ----
results_path = "/content/drive/MyDrive/Models/ArQModel2/eval_results_with_ids.json"

# مجلدات الصور
image_dirs = [
    "/content/drive/MyDrive/COCO/Images/val2014/",
    "/content/drive/MyDrive/COCO/Images/val2014 (1)/"
]

def find_image_path(image_id):
    """
    تبحث عن أي صورة ينتهي اسمها بـ image_id.jpg
    مع ضمان أن image_id = 6 digits (leading zeros)
    """
    # ---- إضافة الأصفار قبل البحث ----
    image_id = str(int(image_id)).zfill(6)

    for folder in image_dirs:
        for fname in os.listdir(folder):
            if fname.endswith(f"{image_id}.jpg"):
                return os.path.join(folder, fname)

    return None


# ---- Load JSON ----
with open(results_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded results:", len(data))
print("Displaying unique image results only...\n")

# ---- تتبع الصور المعروضة ----
shown_image_ids = set()

# ---- Display image + captions (once per image) ----
for item in data:
    img_id = item["image_id"]

    # إذا تكرر الإيميج → تخطي الصورة والكابشنات
    if img_id in shown_image_ids:
        continue

    shown_image_ids.add(img_id)

    img_path = find_image_path(img_id)

    print("=" * 70)
    print(f"Image ID: {img_id}")

    if img_path:
        display(Image(filename=img_path))
    else:
        print("IMAGE NOT FOUND:", img_id)

    print("\nReferences:")
    for j, ref in enumerate(item["reference"]):
        print(f"  [{j+1}] {ref}")

    print("\nGenerated:")
    print(" ", item["generated"])
    print("=" * 70)


In [ ]:
import os
import json
from IPython.display import Image, display

# =========================================================
# Paths
# =========================================================
results_path = "/content/drive/MyDrive/Models/ArQModel2/eval_results_with_ids.json"

# مجلدات الصور (COCO val2014)
image_dirs = [
    "/content/drive/MyDrive/COCO/Images/val2014/",
    "/content/drive/MyDrive/COCO/Images/val2014 (1)/"
]

# =========================================================
# اختاري أيدي الصور التي تريدين عرضها
# - []  → عرض كل الصور (السلوك الافتراضي)
# - [121304] → عرض صورة واحدة
# - [121304, 119364] → عرض عدة صور
# =========================================================
TARGET_IMAGE_IDS = [399921]   # عدّلي هنا حسب الحاجة
# TARGET_IMAGE_IDS = []       # فعّلي هذا السطر لعرض كل الصور

# =========================================================
# Helper: find image path by image_id
# =========================================================
def find_image_path(image_id):
    """
    تبحث عن صورة COCO تنتهي بـ image_id.jpg
    مع ضمان أن image_id مكوّن من 6 أرقام (leading zeros)
    """
    image_id = str(int(image_id)).zfill(6)

    for folder in image_dirs:
        if not os.path.isdir(folder):
            continue
        for fname in os.listdir(folder):
            if fname.endswith(f"{image_id}.jpg"):
                return os.path.join(folder, fname)

    return None

# =========================================================
# Load JSON results
# =========================================================
with open(results_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded results:", len(data))
print("Displaying selected image results...\n")

# =========================================================
# Display image + references + generated (once per image)
# =========================================================
shown_image_ids = set()

for item in data:
    img_id = item.get("image_id")

    # فلترة حسب الأيدي المختارة
    if TARGET_IMAGE_IDS and img_id not in TARGET_IMAGE_IDS:
        continue

    # منع التكرار لنفس الصورة
    if img_id in shown_image_ids:
        continue

    shown_image_ids.add(img_id)

    img_path = find_image_path(img_id)

    print("=" * 70)
    print(f"Image ID: {img_id}")

    if img_path:
        display(Image(filename=img_path))
    else:
        print("IMAGE NOT FOUND:", img_id)

    print("\nReferences:")
    for i, ref in enumerate(item.get("reference", []), 1):
        print(f"  [{i}] {ref}")

    print("\nGenerated:")
    print(" ", item.get("generated", ""))

    print("=" * 70)


In [ ]:
import json
from collections import Counter

json_path = "/content/drive/MyDrive/Models/ArQModel/eval_results_with_ids.json"

# --- Load JSON file ---
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# --- Extract all image_ids ---
image_ids = [item["image_id"] for item in data]

# --- Count duplicates ---
counter = Counter(image_ids)

# --- Filter only repeated IDs ---
duplicates = {img_id: count for img_id, count in counter.items() if count > 1}

# --- Print summary ---
print(f"Total unique image_ids: {len(counter)}")
print(f"Total duplicated image_ids: {len(duplicates)}\n")

# --- Print first 20 duplicates with counts ---
if duplicates:
    print("Some duplicated image_ids with counts:")
    for i, (img_id, count) in enumerate(duplicates.items()):
        print(f"{i+1}. Image ID: {img_id} | Count: {count}")
        if i >= 19:  # show only first 20
            break
else:
    print("No duplicates found.")


In [ ]:
import os, json

results_path = "/content/drive/MyDrive/Models/ArQModel/eval_results_with_ids.json"

# --- Load file ---
with open(results_path, "r") as f:
    data = json.load(f)

print("Total items in file:", len(data))

# --- Detect duplicates ---
seen = set()
duplicates = []

for item in data:
    # Signature = references + generated text
    sig = (tuple(item["reference"]), item["generated"])

    if sig in seen:
        duplicates.append(item)
    else:
        seen.add(sig)

print("Number of duplicates found:", len(duplicates))

# Show first duplicates
if duplicates:
    print("\nShowing first 5 duplicates:\n")
    for i, d in enumerate(duplicates[:100]):
        print(f"Duplicate {i+1}:")
        print("References:")
        for r in d["reference"]:
            print("  -", r)
        print("Generated:", d["generated"])
        print("-" * 40)
else:
    print("No duplicates found.")


In [ ]:
import torch
from models3 import DecoderWithAttention
import torch.serialization

# Allow loading
torch.serialization.add_safe_globals([DecoderWithAttention])

ckpt = torch.load(
    "/content/drive/MyDrive/Models/ArQModel/Checkpoints_best/coco_5_cap_per_img_5_min_word_freq_epoch_025.pth.tar",
    map_location="cpu",
    weights_only=False
)

print("Keys:", ckpt.keys())
print("Has decoder optimizer:", 'decoder_optimizer_state_dict' in ckpt)
print("Has encoder optimizer:", 'encoder_optimizer_state_dict' in ckpt)


In [ ]:
import torch
from models3 import DecoderWithAttention
torch.serialization.add_safe_globals([DecoderWithAttention])

# ضع هنا مسار الـ checkpoint الذي تريدين فحصه
checkpoint_path = "/content/drive/MyDrive/Models/ArQModel/Checkpoints_best/coco_5_cap_per_img_5_min_word_freq_epoch_020.pth.tar"

# تحميل الـ checkpoint
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

# طباعة قيمة BLEU-4
bleu = ckpt.get('bleu-4', None)
print(f"BLEU-4 in this checkpoint: {bleu}")

# تفسير سريع:
if bleu is not None:
    if bleu == 0:
        print("هذا checkpoint من أول التدريب (أوزان تقريباً عشوائية).")
    else:
        print("هذا checkpoint من منتصف التدريب (أوزان مدربة بالفعل).")
else:
    print("لا توجد قيمة BLEU-4 في هذا checkpoint.")


In [ ]:
import json

# ========= نفس الصور (من المودل الأول) =========
SELECTED_IMAGE_IDS = [
    411941,
    399921,
    126192,
    399851,
    315486,
    283698,
    424220,
    295134,
    2142,
    499198,
]

# ========= Path للمودل الثاني =========
json_path = "/content/drive/MyDrive/Models/ArQModel/eval_results_with_ids.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# ========= خريطة image_id → item =========
data_map = {int(x["image_id"]): x for x in data}

print(f"Showing {len(SELECTED_IMAGE_IDS)} images (NOT random)\n")

# ========= عرض بنفس الترتيب =========
for i, img_id in enumerate(SELECTED_IMAGE_IDS, 1):

    if img_id not in data_map:
        print(f"Image ID {img_id} NOT FOUND\n")
        continue

    item = data_map[img_id]

    print(f"Sample {i} | Image ID: {img_id}")
    print("References:")
    for j, ref in enumerate(item["reference"], 1):
        print(f"  Reference {j}: {ref}")

    print("Generated:")
    print(f"  {item['generated']}")
    print("-" * 60)

In [ ]:
import pandas as pd

# ضع مسار الملف الأصلي هنا
input_path = "/content/drive/MyDrive/COCO/Features/trainval_resnet101_faster_rcnn_genome_36.tsv"

# مسار الملف الناتج
output_path = "/content/drive/MyDrive/COCO/Features/sample10.tsv"

df = pd.read_csv(
    input_path,
    sep="\t",
    nrows=10   # أهم سطر
)

df.to_csv(output_path, sep="\t", index=False)

print("تم إنشاء ملف العينة بنجاح")
